In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch5. OpenAI TTS API</font>

# OpenAI API 활용 텍스트-음성 변환(TTS)

- OpenAI API 활용을 통한 입력 텍스트의 자연스러운 음성 변환 구현
- API 키 설정, 기본 TTS 사용법, 고급 기능 활용, 생성 오디오 저장 및 재생 방법

### 튜토리얼 주요 학습 내용

* `python-dotenv` 활용을 통한 OpenAI API 키의 안전한 저장 및 로드 방법
* 간단한 텍스트의 음성 변환 실행 예제 코드
* 음성 모델 선택, 음성 목소리(Voice) 변경, 언어 지원, 속도 조절 등의 고급 기능 활용
* 생성된 음성의 파일 저장 및 Jupyter Notebook 환경 내 재생 방법

# 1. 환경 설정 및 API 클라이언트 인스턴스 생성

* Python 코드 내 `python-dotenv`를 활용한 `.env` 파일 불러오기 프로세스
* `openai.OpenAI()` 클래스를 이용한 API 클라이언트 인스턴스 생성 방법
* API 키의 명시적 전달 또는 `OPENAI_API_KEY` 환경 변수 사전 설정의 필요성
* 환경 변수 또는 인자 미지정 시 오류 발생
    - `OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable` 

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

```
위 코드에서는 .env에서 불러온 API 키로 client 객체를 생성했습니다. 이제 이 client를 통해 DALL-E를 비롯한 OpenAI API 요청을 보낼 수 있습니다.
```

> 참고로, api_key를 생략하면 OPENAI_API_KEY 환경 변수를 자동으로 참조합니다.


# 2. 기본 TTS 변환
- 텍스트를 음성으로 간단히 만들기

- **tts-1** 모델 : 음성의 종류(voice)를 선택 가능

- [OpenAI TTS](https://developers.openai.com/api/docs/guides/text-to-speech)



In [6]:
# 변환할 텍스트 - with 구문을 사용하여 TTS 음성 생성 응답을 스트리밍 방식으로 받아오기
text = 'Hello, OpenAI! This is a text to speech test.'
with client.audio.speech.with_streaming_response.create(
    model="tts-1",
    voice="ash", # 사용하고자 하는 목소리 (ash, alloy, echo, fable, onyx, nova, shimmer)
    input=text,
)  as response:
    response.stream_to_file('data/ch5_outputash.mp3')

In [7]:
from IPython.display import Audio
Audio('data/ch5_outputash.mp3', autoplay=True)

In [8]:
response = client.audio.speech.create(
    model="tts-1",
    voice="alloy",
    input=text
)

* OpenAI TTS API(`openai_client.audio.speech.create`)의 주요 매개변수 및 응답 데이터 처리 구조

- 매개변수
    * `model="tts-1"` : 실시간 응답에 최적화된 OpenAI의 기본 TTS 모델 지정
    * `voice="nova"` : 출력 음성용 프리셋 목소리 선택
    * `input=text` : 음성으로 변환할 텍스트 문자열 전달
    
* API 호출 및 응답 객체 반환 : 요청 송신 후 이진 형식의 오디오 데이터를 포함한 내부 응답 객체(`response`) 수신

* `response.content` 속성 : 내부 응답 객체로부터 실제 오디오 바이너리 데이터 추출

In [11]:
audio_data = response.content
print(f'입력 텍스트의 글자 수 : {len(text)} 개')
print(f'생성된 오디오 데이터 크기 : {len(audio_data)} 바이트')

입력 텍스트의 글자 수 : 45 개
생성된 오디오 데이터 크기 : 43392 바이트


In [13]:
with open('data/ch5_alloy.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

In [14]:
Audio('data/ch5_alloy.mp3', autoplay=True)

```
위와 같이 response.content를 사용하면 음성 데이터의 바이너리를 얻을 수 있고, len(audio_data)로 바이트 크기를 확인할 수 있습니다. 이제 텍스트가 음성 데이터로 변환된 것을 확인했으니, 다음 단계에서 다양한 고급 기능을 활용하는 방법을 알아보겠습니다.

```

# 3. 고급 기능 활용: 모델 선택, 목소리 변경, 언어 지원

- 다양한 목소리(voice) 선택 : 남성/여성 및 톤이 다른 여러 프리셋 음성
- 모델 선택 : 기본 모델 tts-1 외에 고품질 모델 tts-1-hd 사용법
- 다국어 지원 : 영어 이외의 언어로도 TTS가 가능 (한국어, 스페인어 등)
- 음성 속도 조절 : speed 파라미터로 발음 속도를 빠르게 또는 느리게 조절

## 3.1 여러 가지 음성 목소리 선택하기

**프리셋 목소리(voice)** : 6가지 목소리를 제공
- Alloy – (예상: 남성적이고 부드러운 음색)
- Echo – (맑고 청아한 톤)
- Fable – (동화 구연 같은 따뜻한 톤)
- Nova – (친근한 여성 음색으로 추정)
- Onyx – (깊고 진중한 남성 음색)
- Shimmer – (밝고 생동감 있는 여성 음색)

```
각 목소리는 영어를 기반으로 최적화되어 있지만, 다국어 텍스트도 발음할 수 있습니다. 원하는 분위기나 성별 느낌에 따라 위 이름들 중 선택하면 됩니다. 사용 방법은 간단히 voice 파라미터에 해당 이름 문자열을 지정하는 것입니다.
```


In [17]:
# 여러가지 목소리
response_diff_voice = client.audio.speech.create(
    model="tts-1",
    voice="shimmer",
    input="this text will be spoken with different voices.",
    speed=1.8
)

audio_data = response_diff_voice.content
print('음성 변환 데이터 크기 :', len(audio_data))

with open('data/ch5_Shimmer.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

음성 변환 데이터 크기 : 26496


In [18]:
Audio('data/ch5_Shimmer.mp3', autoplay=True)

```
위 코드에서는 voice="shimmer"로 설정하여 Shimmer 목소리로 텍스트를 음성 변환했습니다. response_diff_voice.content에 변환된 오디오 데이터가 들어있으며, 바이트 크기를 출력하여 데이터가 생성되었음을 확인했습니다. 나중에 오디오를 재생해보면, 처음 사용한 Nova 목소리와 Shimmer 목소리가 다르게 들리는 것을 알 수 있습니다.
```

> Tip: 음성 이름은 대소문자를 구분하지 않는 소문자로 입력합니다. 또한, OpenAI에서 새로운 목소리를 추가로 제공할 수 있으므로, 최신 정보는 OpenAI 공식 문서나 발표를 참고하세요. (예: 추가로 Ash, Coral, Sage 등의 목소리가 제공되기 시작할 수 있습니다.)


## 3.2 TTS 모델 선택: 실시간 vs 고품질

- OpenAI 제공 TTS 모델 구분 및 특성 비교 (tts-1 vs tts-1-hd)
    - tts-1 모델 : 빠른 응답 속도 중심의 실시간 상호작용 최적화 기본 모델
    - tts-1-hd 모델 : 응답 속도 대비 고품질 음성 생성 중심의 품질 최적화 모델

- 모델 선택 기준 : 응답 속도와 음질 간 트레이드오프(Trade-off) 고려 및 용도별 모델 선택 (실시간 서비스: tts-1, 음성 품질 우선 서비스: tts-1-hd)


> 참고 : OpenAI TTS의 요금은 입력 문자 수를 기준으로 부과되며, 1000자당 0.015 달러부터 시작합니다. 고품질 모델을 쓴다고 해서 추가 요금이 부과되는지는 공식 가격 정책을 참고해야 하지만, 일반적으로 더 많은 연산을 수행하므로 속도 면에서만 차이가 있습니다.

## 3.3 다국어 텍스트 변환하기

In [20]:
# 한국어 텍스트를 음성으로 변환
kr_txt = '안녕하세요. 저는 오픈 API로 생성된 한국어 화자입니다.'
response_ko = client.audio.speech.create(
    model = 'tts-1-hd',
    voice = 'shimmer',
    input = kr_txt,
    speed=0.9
)

audio_data = response_ko.content
print('음성 오디오 데이터 크기 :', len(audio_data))

with open('data/ch5_Shimmer_ko_hd.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

음성 오디오 데이터 크기 : 74880


In [21]:
Audio('data/ch5_Shimmer_ko_hd.mp3', autoplay=True)

```
이 코드는 한국어 문장을 받아 Shimmer 목소리로 음성을 생성합니다. speed=0.8로 설정하여 기본 속도의 80%로 조금 느리게 발음되도록 했습니다. 이렇게 하면 한국어 발음이 너무 빠르지 않게 출력되어 듣기 더 편할 수 있습니다. 

OpenAI TTS는 Whisper 모델의 다국어 지원을 따르므로, 한국어 뿐만 아니라 여러 언어를 지원합니다. Whisper 모델이 잘 동작하는 언어들은 TTS에서도 무리 없이 사용 가능하며, 예를 들면 스페인어 문장을 넣으면 스페인어 발음의 음성이 나옵니다. 다만 현재의 목소리들은 주로 영어 억양 기반이라서, 다른 언어를 발음하더라도 약간의 억양 특징이 섞일 수 있습니다. 그럼에도 불구하고 한국어 등 꽤 자연스럽게 들리는 것을 확인할 수 있습니다.
```

>팁: 만약 특정 언어로 발음된 음성을 얻을 때 발음이 어색하다면, 문자를 발음 나는 대로 표기하거나, 문장 부호로 억양을 조절해보는 방법도 있습니다. 예를 들어 영어 문장에서도 쉼표, 마침표 등을 넣으면 잠시 멈춤이 생겨 더 자연스럽게 들릴 수 있습니다. TTS 시스템은 입력 텍스트를 그대로 읽기 때문에, 철자가 아닌 발음을 유도하려면 입력을 조정하는 트릭을 쓸 수도 있습니다.

## 3.4 발음 속도 조절하기

speed : 0.25 ~ 4 ; 기본값은 1.0 (보통 속도)

- speed=1.0 : 기본 속도 (보통 사람 말하기 속도)
- speed<1.0 : 느린 발음 (예: 0.5는 절반 속도)
- speed>1.0 : 빠른 발음 (예: 2.0은 두 배 속도로 빨리 말함)

In [34]:
kr_txt = '안녕하세요. 저는 오픈 API로 생성된 한국어 화자입니다.'
response_ko = client.audio.speech.create(
    model = 'tts-1-hd',
    voice = 'fable',
    input = kr_txt,
    speed=2.0
)

audio_data = response_ko.content
print('음성 오디오 데이터 크기 :', len(audio_data))

with open('data/ch5_Shimmer_ko_hd.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

음성 오디오 데이터 크기 : 34176


In [23]:
Audio('data/ch5_Shimmer_ko_hd.mp3', autoplay=True)

```
위 코드에서는 speed=2.0으로 설정하여 Fable 목소리가 평소보다 두 배 빠르게 말하도록 했습니다. 나중에 이 오디오를 들어보면 같은 문장이라도 매우 빠르게 읽히는 것을 확인할 수 있습니다. 이처럼 상황에 따라 속도를 조절하면 긴 문장을 짧은 시간에 듣거나, 어린아이에게 들려주듯 천천히 말하게 하는 등 다양한 활용이 가능합니다. 

지금까지 OpenAI TTS의 여러 기능을 사용하여 텍스트를 음성으로 변환하는 방법을 살펴보았습니다. 마지막으로, 이렇게 생성된 음성 데이터를 파일로 저장하고 실제로 들어보는 방법을 알아보겠습니다.
```


# 4. 오디오 파일 저장 및 Jupyter에서 재생하기
* TTS API 응답 데이터 형태 : 메모리 상 바이너리 데이터(`bytes`) 형태의 음성 데이터 수신
* 오디오 파일 저장 및 재생 : 바이너리 데이터를 통한 파일(예: MP3) 저장 및 일반 오디오 플레이어 재생 지원
* Jupyter Notebook 오디오 위젯 활용 : 저장 파일 로드를 통한 Jupyter Notebook 내 오디오 위젯 재생 가능

## 4.1 오디오 데이터 파일로 저장하기
* OpenAI TTS 기본 응답 포맷 : 기본 설정 시 MP3 형식 오디오 데이터 반환
* 기본 포맷 파일 저장 방식 : 수신된 바이너리 데이터를 `.mp3` 확장자 파일로 직접 저장
* `response.content` 활용 파일 저장 : 응답 객체 내 `content` 속성을 활용한 오디오 바이너리 데이터 파일 저장

In [24]:
kr_txt = '안녕하세요. 저는 오픈 API로 생성된 한국어 화자입니다.'
response_ko = client.audio.speech.create(
    model = 'tts-1-hd',
    voice = 'fable',
    input = kr_txt,
    speed=2.0,
    response_format='wav'
)

audio_data = response_ko.content
print('음성 오디오 데이터 크기 :', len(audio_data))

with open('data/ch5_Shimmer_ko_hd.mp3', 'wb') as audio_file:
    audio_file.write(audio_data)

음성 오디오 데이터 크기 : 99344


```
파일 저장은 일반적인 파일 쓰기 방식과 동일합니다. with open("파일경로", "wb") as f:로 바이너리 쓰기 모드("wb")로 파일을 열고, f.write(오디오데이터)로 내용을 기록합니다. 이렇게 하면 현재 디렉토리에 output_fast.mp3라는 파일이 생성되고, 변환된 음성이 그 안에 저장됩니다. OpenAI Python 라이브러리는 편의를 위해 스트리밍 저장 함수도 제공합니다. response_fast와 같은 응답 객체는 stream_to_file("경로")라는 메소드를 갖고 있어서 한 줄로 바로 파일 저장을 할 수도 있습니다. 예를 들어 response_fast.stream_to_file("output_fast.mp3")라고 하면 같은 결과를 얻습니다. 이 방법은 내부적으로 데이터를 조금씩 쓰는 스트림 방식을 사용하여 메모리 효율성을 높일 수 있습니다.
```

> 참고 : OpenAI TTS API는 기본 mp3 외에도 원하면 다른 오디오 포맷으로 받을 수 있습니다. response_format 파라미터에 "wav", "aac", "flac" 등 지원되는 포맷을 지정하면 해당 형식으로 반환받을 수 있습니다. 예를 들어 openai_client.audio.speech.create(..., response_format="wav")로 요청하면 response.content가 WAV 포맷의 데이터로 구성됩니다. 일반적인 사용에서는 mp3로 충분하겠지만, 필요에 따라 포맷을 선택하세요.

## 4.2 Jupyter Notebook에서 오디오 재생하기

* 저장된 오디오 파일 재생 단계 : 바이너리 데이터 저장 완료 후 오디오 파일 재생 수행
* `IPython.display.Audio` 클래스 활용 : Jupyter Notebook 환경 내 direct 오디오 재생을 위한 `IPython.display` 모듈 제공 클래스 활용
* 브라우저 기반 오디오 플레이어 위젯 제공 : 웹 브라우저 상 오디오 플레이어 위젯 출력 및 오디오 재생 기능 지원

In [25]:
Audio('data/ch5_Shimmer_ko_hd.mp3', autoplay=True)

```
위 코드를 실행하면 Jupyter Notebook 출력 영역에 오디오 컨트롤 바가 나타날 것입니다. autoplay=True로 설정했으므로 자동으로 재생도 시작됩니다. (브라우저 정책에 따라 자동 재생이 막힐 수 있는데, 그 경우 수동으로 재생 버튼을 누르면 됩니다.) 이제 TTS로 생성한 음성을 실제로 들어볼 수 있습니다. 

만약 메모리의 데이터로 바로 재생하고 싶다면, Audio(data=audio_bytes, rate=...) 형태로도 가능합니다. 그러나 TTS의 결과는 이미 압축된 오디오 파일 형태이므로, filename을 통해 재생하는 편이 간편합니다. 여러 개의 음성을 비교하고 싶다면 다른 파일명으로 저장한 후 각각 Audio(...)로 불러와 보면 됩니다. 

마지막으로, Jupyter가 아닌 환경(예: 백엔드 서버)에서 TTS를 활용하는 경우에는 이렇게 파일로 저장한 후, 사용자에게 그 파일을 제공하거나 스트리밍 전송하여 들려줄 수 있습니다.
```

## 4.3 재생 완료한 오디오 파일 삭제하기

In [27]:
import os
try:
    os.remove('data/ch5_Shimmer_ko_hd.mp3')
    print('삭제 완료')
except Exception as e:
    print('이미 삭제된 파일입니다.', e)

삭제 완료


In [29]:
import os
try:
    os.remove('data/ch5_Shimmer.mp3')
    print('삭제 완료')
except Exception as e:
    print('이미 삭제된 파일입니다.', e)

이미 삭제된 파일입니다. [WinError 2] 지정된 파일을 찾을 수 없습니다: 'data/ch5_Shimmer.mp3'


In [30]:
import os
try:
    os.remove('data/ch5_alloy.mp3')
    print('삭제 완료')
except Exception as e:
    print('이미 삭제된 파일입니다.', e)

삭제 완료


In [33]:
import os
try:
    os.remove('data/ch5_outputash.mp3')
    print('삭제 완료')
except Exception as e:
    print('이미 삭제된 파일입니다.', e)

삭제 완료
